In [0]:
# Union history and live tables
# Perform the group by the hour

In [0]:
mode = dbutils.widgets.get("mode")

In [0]:
%sql
CREATE TABLE IF NOT EXISTS gold_fact_hourlyprices_last90days(
  date_key STRING,
  time_key STRING,
  currency_key STRING,
  coin_key STRING,
  prices DOUBLE,
  market_caps DOUBLE,
  total_volumes DOUBLE
)
USING DELTA
LOCATION 'abfss://gold@bitcoindatalake.dfs.core.windows.net/fact_tables/gold_fact_hourlyprices_last90days/'

In [0]:
%sql
CREATE TABLE IF NOT EXISTS gold_fact_4hourlyohlc_last30days(
  date_key STRING,
  time_key STRING,
  currency_key STRING,
  coin_key STRING,
  open DOUBLE,
  high DOUBLE,
  low DOUBLE,
  close DOUBLE
)
USING DELTA
LOCATION 'abfss://gold@bitcoindatalake.dfs.core.windows.net/fact_tables/gold_fact_4hourlyohlc_last30days/'

In [0]:
%sql
BEGIN
  IF :mode == 'incremental_load' THEN
    CREATE TABLE IF NOT EXISTS gold_fact_30minohlc_lastday(
      date_key STRING,
      time_key STRING,
      currency_key STRING,
      coin_key STRING,
      open DOUBLE,
      high DOUBLE,
      low DOUBLE,
      close DOUBLE
    )
    USING DELTA
    LOCATION 'abfss://gold@bitcoindatalake.dfs.core.windows.net/fact_tables/gold_fact_30minohlc_lastday/';
  END IF;
END;

In [0]:
%sql
BEGIN
  IF :mode == 'incremental_load' THEN
    CREATE TABLE IF NOT EXISTS gold_fact_5minprices_lastday(
    date_key STRING,
    time_key STRING,
    currency_key STRING,
    coin_key STRING,
      prices DOUBLE,
      market_caps DOUBLE,
      total_volumes DOUBLE

    )
    USING DELTA
    LOCATION 'abfss://gold@bitcoindatalake.dfs.core.windows.net/fact_tables/gold_fact_5minprices_lastday/';
  END IF;
END;

In [0]:
%sql
CREATE TABLE IF NOT EXISTS gold_dim_currency(
  currency_code STRING,
  currency_key STRING PRIMARY KEY,
  currency_name STRING,
  currency_symbol STRING,
  is_active BOOLEAN
)
USING DELTA
LOCATION 'abfss://gold@bitcoindatalake.dfs.core.windows.net/dim_tables/gold_dim_currency_table/'

In [0]:
%sql
CREATE TABLE IF NOT EXISTS gold_dim_coin(
  coin_id STRING,
  coin_key STRING PRIMARY KEY,
  founded_year Date,
  ticker_symbol STRING,
  `source` STRING,
  is_active BOOLEAN
)
USING DELTA
LOCATION 'abfss://gold@bitcoindatalake.dfs.core.windows.net/dim_tables/gold_dim_coins_table/'

In [0]:
%sql
CREATE TABLE IF NOT EXISTS gold_dim_date(
  date_key String PRIMARY KEY,
  full_date Date,
  day_of_week STRING,
  day_number_in_week Integer,
  month_name STRING,
  `quarter` INTEGER,
  `year` INTEGER,
  is_weekend BOOLEAN
)
USING DELTA
LOCATION 'abfss://gold@bitcoindatalake.dfs.core.windows.net/dim_tables/gold_dim_date_table/'

In [0]:
%sql
CREATE TABLE IF NOT EXISTS gold_dim_time(
  time_key String PRIMARY KEY,
  time_of_day STRING,
  `hour` INT,
  `minute` INT,
  period_name String,
  trading_session String,
  is_4h_interval BOOLEAN,
  is_1h_interval BOOLEAN,
  is_30m_interval BOOLEAN
  )
USING DELTA
LOCATION 'abfss://gold@bitcoindatalake.dfs.core.windows.net/dim_tables/gold_dim_time_table/'

In [0]:
%sql
BEGIN
  IF :mode == 'initial_load' THEN
MERGE INTO gold_dim_coin as target
USING(
  SELECT *
  FROM silver_coin
) as source
on target.coin_key = source.coin_key
WHEN MATCHED THEN
UPDATE SET *
WHEN NOT MATCHED THEN
INSERT *;
END IF;
END;

In [0]:
%sql
BEGIN
  IF :mode == 'initial_load' THEN
MERGE INTO gold_dim_currency as target
USING(
  SELECT *
  FROM silver_currency
) as source
on target.currency_key = source.currency_key
WHEN MATCHED THEN
UPDATE SET *
WHEN NOT MATCHED THEN
INSERT *;
END IF;
END;

In [0]:
%sql
MERGE INTO gold_dim_date as target
USING(
with ordered_dates_CTE(
SELECT 
DISTINCT CAST(datetime as DATE) as full_date
FROM silver_hourlyprices_last90days
UNION SELECT 
DISTINCT CAST(datetime as DATE) as full_date
FROM silver_5minprices_lastday
ORDER BY full_date
)
select
CAST(date_format(full_date, 'yyyyMMdd') as STRING) as date_key,
full_date,
date_format(full_date, 'EEEE') as day_of_week,
dayofweek(full_date) as day_number_in_week,
date_format(full_date, 'MMMM') as month_name,
date_format(full_date, 'Q') as `quarter`,
date_format(full_date, 'yyyy') as `year`,
case
    when day_of_week = 'Sat' or day_of_week = 'Sun' THEN true
    ELSE false
END as is_weekend
FROM ordered_dates_CTE
) as source
on target.date_key = source.date_key
WHEN MATCHED THEN
UPDATE SET *
WHEN NOT MATCHED THEN
INSERT *

In [0]:
%sql
BEGIN
  IF :mode == 'initial_load' THEN
MERGE INTO gold_dim_time as target
USING(
WITH base_sequence AS (
  SELECT explode(sequence(0, 1435, 5)) AS minutes_from_midnight
)
SELECT
    -- Key as INT for performance
    CAST(minutes_from_midnight AS INT) as time_key,
    
    -- Formatting
    printf('%02d:%02d', floor(minutes_from_midnight / 60), minutes_from_midnight % 60) AS time_of_day,
    floor(minutes_from_midnight / 60) AS hour,
    minutes_from_midnight % 60 AS minute,
    
    -- AM/PM logic
    CASE WHEN floor(minutes_from_midnight / 60) < 12 THEN 'AM' ELSE 'PM' END AS am_pm,
    
    -- Period & Sessions
    CASE 
        WHEN floor(minutes_from_midnight / 60) BETWEEN 0 AND 5 THEN 'Night'
        WHEN floor(minutes_from_midnight / 60) BETWEEN 6 AND 11 THEN 'Morning'
        WHEN floor(minutes_from_midnight / 60) BETWEEN 12 AND 17 THEN 'Afternoon'
        ELSE 'Evening'
    END AS period_name,
    
    -- Trading Sessions (NY: 8am-5pm EST, London: 3am-12pm EST)
    CASE 
        WHEN floor(minutes_from_midnight / 60) BETWEEN 8 AND 12 THEN 'London-NY Overlap'
        WHEN floor(minutes_from_midnight / 60) BETWEEN 3 AND 12 THEN 'London'
        WHEN floor(minutes_from_midnight / 60) BETWEEN 8 AND 17 THEN 'New York'
        ELSE 'Other'
    END AS trading_session,

    -- Interval Flags
    (floor(minutes_from_midnight / 60) IN (0, 4, 8, 12, 16, 20) AND (minutes_from_midnight % 60) = 0) AS is_4h_interval,
    (minutes_from_midnight % 60) = 0 AS is_1h_interval,
    (minutes_from_midnight % 60) IN (0, 30) AS is_30m_interval
FROM base_sequence
) as source
on target.time_key = source.time_key
WHEN MATCHED THEN
UPDATE SET *
WHEN NOT MATCHED THEN
INSERT *;
END IF;
END;

In [0]:
if mode == "incremental_load":
    print("Running Gold Table Overwrite...")
    
    query = """
    INSERT OVERWRITE gold_fact_30minohlc_lastday
    WITH snapped_source AS (
            SELECT 
                *,
                -- Round the Unix timestamp to the nearest 1800 seconds (30 minutes)
                FROM_UNIXTIME(ROUND(UNIX_TIMESTAMP(datetime) / 1800) * 1800) as snapped_datetime
            FROM bitcoinproject_workspace.default.silver_30minohlc_lastday
        )
        SELECT 
            CAST(DATE_FORMAT(snapped_datetime, 'yyyyMMdd') AS INT) as date_key,
            -- time_key will now strictly be multiples of 30 (0, 30, 60, 90...)
            (HOUR(snapped_datetime) * 60) + MINUTE(snapped_datetime) as time_key,
            '1' as currency_key,
            '1' as coin_key,
            open,
            high,
            low,
            close
        FROM snapped_source
    """
    
    spark.sql(query)
else:
    print("Skipping Gold Overwrite (Initial Load Mode)")

In [0]:
if mode == "incremental_load":
    print("Running Gold Table Overwrite...")
    
    query = """
    INSERT OVERWRITE gold_fact_5minprices_lastday
    WITH snapped_source AS (
            SELECT 
                *,
                -- Round the Unix timestamp to the nearest 300 seconds (5 minutes)
                FROM_UNIXTIME(ROUND(UNIX_TIMESTAMP(datetime) / 300) * 300) as snapped_datetime
            FROM bitcoinproject_workspace.default.silver_5minprices_lastday
        )
        SELECT 
            CAST(DATE_FORMAT(snapped_datetime, 'yyyyMMdd') AS INT) as date_key,
            -- time_key will now strictly be multiples of 5
            (HOUR(snapped_datetime) * 60) + MINUTE(snapped_datetime) as time_key,
            '1' as currency_key,
            '1' as coin_key,
            prices,
            market_caps,
            total_volumes
        FROM snapped_source
        
    """
    
    spark.sql(query)
else:
    print("Skipping Gold Overwrite (Initial Load Mode)")

In [0]:
%sql
MERGE INTO gold_fact_hourlyprices_last90days as target
USING (
    WITH hourly_aggregated_CTE AS (
        SELECT
            DATE_TRUNC('hour', datetime) AS hour_start,
            AVG(prices) AS prices,
            AVG(market_caps) AS market_caps,
            AVG(total_volumes) as total_volumes
        FROM bitcoinproject_workspace.default.silver_5minprices_lastday
        GROUP BY 1
    ), 
    combined_source AS (
        SELECT datetime, prices, market_caps, total_volumes FROM silver_hourlyprices_last90days
        UNION ALL
        SELECT hour_start, prices, market_caps, total_volumes FROM hourly_aggregated_CTE
    ),
    snapped_source AS (
        SELECT 
            *,
            FROM_UNIXTIME(ROUND(UNIX_TIMESTAMP(datetime) / 3600) * 3600) as snapped_datetime
        FROM combined_source
    ),

    final_dedup AS (
        SELECT 
            *,
            ROW_NUMBER() OVER (
                PARTITION BY DATE_TRUNC('hour', snapped_datetime), 
                             (HOUR(snapped_datetime) * 60) + MINUTE(snapped_datetime)
                ORDER BY datetime DESC 
            ) as rank
        FROM snapped_source
    )

    SELECT 
        CAST(DATE_FORMAT(s.snapped_datetime, 'yyyyMMdd') AS INT) as date_key,
        -- 2. Calculate time_key from the already rounded/snapped datetime
        (HOUR(s.snapped_datetime) * 60) + MINUTE(s.snapped_datetime) as time_key,
        '1' as currency_key,
        '1' as coin_key,
        s.prices,
        s.market_caps,
        s.total_volumes
    FROM final_dedup s
    WHERE rank = 1
) as source
ON target.date_key = source.date_key 
   AND target.time_key = source.time_key
WHEN MATCHED THEN
  UPDATE SET 
  target.currency_key = source.currency_key,
    target.coin_key = source.coin_key,
    target.prices = source.prices,
    target.market_caps = source.market_caps,
    target.total_volumes = source.total_volumes
WHEN NOT MATCHED THEN
  INSERT (date_key, time_key,currency_key, coin_key, prices, market_caps, total_volumes)
  VALUES (source.date_key, source.time_key,source.currency_key, source.coin_key, source.prices, source.market_caps, source.total_volumes);

In [0]:
%sql
MERGE INTO gold_fact_4hourlyohlc_last30days as target
USING (
    WITH 4hours_aggregated_CTE AS (
        SELECT
            DATE_TRUNC('hour', datetime) AS hour_start,
            FIRST(open) as open,
            MAX(high) as high,
            MIN(low) as low,
            LAST(close) as close
        FROM bitcoinproject_workspace.default.silver_4hourlyohlc_last30days
        GROUP BY 1
    ),
    combined_source AS (
        -- Combine history with ONLY the 4-hour intervals from live data
        SELECT datetime, open, high, low, close 
        FROM bitcoinproject_workspace.default.silver_30minohlc_lastday
        -- FILTER: Only keep hours 0, 4, 8, 12, 16, 20 AND only the top of the hour (:00)
        WHERE HOUR(datetime) % 4 = 0 AND MINUTE(datetime) = 0
        
        UNION ALL
        
        SELECT hour_start, open, high, low, close 
        FROM 4hours_aggregated_CTE
    ),
    snapped_source AS (
        SELECT 
            *,
            -- Round the Unix timestamp to the nearest 300 seconds (5 minutes)
            -- This ensures 00:01:23 becomes 00:00:00 and 00:04:45 becomes 00:05:00
            FROM_UNIXTIME(ROUND(UNIX_TIMESTAMP(datetime) / 300) * 300) as snapped_datetime
        FROM combined_source
    ),
    final_dedup AS (
        SELECT 
            *,
            -- Partition by the SNAPPED time to ensure we only have one record per 5-min interval
            ROW_NUMBER() OVER (
                PARTITION BY DATE_TRUNC('hour', snapped_datetime), 
                             (HOUR(snapped_datetime) * 60) + MINUTE(snapped_datetime)
                ORDER BY datetime DESC -- Keep the latest actual data point for that interval
            ) as rank
        FROM snapped_source
    )
    SELECT 
        CAST(DATE_FORMAT(s.snapped_datetime, 'yyyyMMdd') AS INT) as date_key,
        -- Calculate time_key: (Hour * 60) + Minute (always a multiple of 5 now)
        (HOUR(s.snapped_datetime) * 60) + MINUTE(s.snapped_datetime) as time_key,
        '1' as currency_key,
        '1' as coin_key,
        s.open,
        s.high,
        s.low,
        s.close
    FROM final_dedup s
    WHERE rank = 1
) as source
ON target.date_key = source.date_key 
   AND target.time_key = source.time_key
   AND target.coin_key = source.coin_key
   AND target.currency_key = source.currency_key

WHEN MATCHED THEN
  UPDATE SET 
    target.open = source.open,
    target.high = source.high,
    target.low = source.low,
    target.close = source.close

WHEN NOT MATCHED THEN
  INSERT (date_key, time_key, currency_key, coin_key, open, high, low, close)
  VALUES (source.date_key, source.time_key, source.currency_key, source.coin_key, source.open, source.high, source.low, source.close);